# Snap rounding: make an intersecting arrangement work on a fixed grid

This lab exposes the hot-pixel steps of a small exact-arithmetic teaching noder. The algorithm computes arrangement vertices first, activates grid cells, then reroutes segments through hot pixels they intersect. It is not a production-scale implementation. Read the [method lesson](../algorithms/geometry/snap-rounding.md) and [package contract](../implementations/python/snap-rounding-noder/README.md).

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

package = Path.cwd() / 'implementations' / 'python' / 'snap-rounding-noder'
sys.path.insert(0, str(package))
from snap_rounding import snap_round


## 1. Start with two crossing segments

The inputs are synthetic. The exact intersection is calculated before the hot pixel is selected. Try changing the endpoints or grid size in the next cell.

In [ ]:
segments = [((0, 0), (3, 3)), ((0, 2), (3, 0))]
grid_size = 1
result = snap_round(segments, grid_size=grid_size)
print('Hot pixels:', result.hot_pixels)
print('Snapped polylines:', result.polylines)
print('Noded edges:', result.edges)


## 2. Draw the original segments, hot pixels and snapped polylines

A hot pixel is a grid cell activated by an arrangement vertex. Every segment intersecting that cell is rerouted through its centre. The shaded squares below show the closed pixel cells used by this package.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for x in range(-1, 5):
    ax.axvline(x, linewidth=0.5, alpha=0.35)
    ax.axhline(x, linewidth=0.5, alpha=0.35)
for i, segment in enumerate(segments):
    (x1, y1), (x2, y2) = segment
    ax.plot([x1, x2], [y1, y2], linestyle='--', linewidth=1.5, label=f'Input {i + 1}')
for x, y in result.hot_pixels:
    ax.add_patch(Rectangle((x - 0.5, y - 0.5), 1, 1, alpha=0.2))
for i, line in enumerate(result.polylines):
    if line:
        xs, ys = zip(*line)
        ax.plot(xs, ys, marker='o', linewidth=2, label=f'Snapped {i + 1}')
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Hot-pixel snap rounding')
ax.legend()
plt.show()


## 3. Change the grid spacing

A coarser grid can map distinct arrangement vertices to the same hot-pixel centre. A finer grid may preserve more detail. Neither choice is automatically correct: choose a grid in relation to the accuracy and scale of the application.

In [ ]:
for spacing in (0.5, 1, 2):
    fitted = snap_round(segments, grid_size=spacing)
    print(f'grid size={spacing}: {len(fitted.hot_pixels)} hot pixels')
    print(fitted.hot_pixels)
    print(fitted.polylines)


## 4. Inspect degeneracies

Collinear overlaps, endpoint contacts and segments lying exactly on a pixel boundary deserve explicit tests. The implementation uses closed pixel squares, so a boundary segment can meet both adjacent cells. Its deterministic ordering makes output reproducible; it does not establish every formal topological guarantee of a production noder.

In [ ]:
cases = {
    'endpoint contact': [((0, 0), (1, 0)), ((1, 0), (1, 1))],
    'collinear overlap': [((0, 0), (4, 0)), ((1, 0), (3, 0))],
    'negative half-grid tie': [((-0.5, -0.5), (0.5, 0.5))],
    'boundary-aligned segment': [((0, 0.5), (3, 0.5)), ((1, 0), (1, 1))],
}
for label, inputs in cases.items():
    fitted = snap_round(inputs)
    print('\n', label)
    print('hot pixels:', fitted.hot_pixels)
    print('polylines:', fitted.polylines)


## 5. Questions to take away

1. Why must the exact arrangement vertices be found before rounding?
2. Which segments are rerouted through a hot pixel?
3. What changes when the grid becomes coarser?
4. Why does a closed pixel boundary need an explicit tie policy?
5. Which performance bottlenecks would an output-sensitive implementation address?

Complete the [snap-rounding exercise](../exercises/snap-rounding-noder.md). Compare the teaching implementation with a production implementation and record the differences in precision, topology and performance guarantees.